# 最終演習Y-MI：Sena et al. 2025（水接触角の機械学習）を再現する
教科書の YM-5（手順）と同じ流れです。上から順に実行してください。Colab では「ランタイム → すべてのセルを実行」でも動きますが、SI の PDF を用意するセルでアップロードの欄が出て、ファイルを選ぶまで止まって待ちます。止めずに合成データで最後まで流すなら、そのセルの `USE_UPLOAD = True` を `False` にしてから実行します（入れ子の交差検証は最後の任意のセル）。

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q rdkit xgboost pdfplumber
    !apt-get -qq install -y fonts-noto-cjk > /dev/null
import matplotlib.font_manager as fm
# 図は run() で動かす別のプロセスがかくので、Colab ではフォントの一覧のキャッシュを作り直してから確かめる
fonts = fm._load_fontmanager(try_read_cache=False) if IN_COLAB else fm.fontManager
if not {f.name for f in fonts.ttflist} & {"Hiragino Sans", "Noto Sans CJK JP"}:   # set_jp_font と同じ候補
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。"
          "Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください")
print("Colab" if IN_COLAB else "ローカルの Jupyter", sys.version.split()[0])

## スクリプトを書き出す（教科書の YM-6 と同じ内容）

In [ ]:
%%writefile ymi_core.py
"""最終演習Y-MI の共通の部品（読み込み・評価の指標・LOOCV・モデル・物理のベースライン）。

ほかのスクリプト（reproduce_ymi.py など）が import して使う。単独では実行しない。
"""
import json
from pathlib import Path

import numpy as np
import pandas as pd

# 表 S4 の並び（F_1 = 高分子の番号 0〜25）
ABBR = ["PEEK", "PPS", "PSU", "PC", "PE", "PP", "PET", "PMMA", "PEO", "ABS",
        "FEP", "NYLON6", "PBD", "PCTFE", "PIB", "PS", "PTFE", "PVA", "PVOH",
        "PVC", "PVF", "PVDF", "PES", "PPO", "SA", "PBT"]

# 特徴量の名前（論文の表2）。F_2・F_3 は実験値、F_4 は MD、F_5〜F_15 は RDKit
FEAT = {
    "F_2": "表面エネルギー", "F_3": "粗さSa", "F_4": "ΔGsolv(MD)",
    "F_5": "sp3原子の数", "F_6": "sp2原子の数", "F_7": "sp3炭素の割合",
    "F_8": "水素結合の受容体", "F_9": "水素結合の供与体", "F_10": "LogP",
    "F_11": "水素以外の原子の数", "F_12": "分子量", "F_13": "水素を除く分子量",
    "F_14": "水素原子の数", "F_15": "価電子の数",
}
MODEL1 = [f"F_{i}" for i in range(2, 16)]   # 14個（実験値＋計算値）
MODEL2 = [f"F_{i}" for i in range(4, 16)]   # 12個（計算値だけ）

# 論文の表4（Model 1・Model 2）と表 S5（既定のハイパーパラメータの XGBoost）
PAPER = {
    "model1": {"rmse": 5.7, "mae": 3.7, "r2": 0.86, "slope": 0.75, "icpt": 21.4},
    "model2": {"rmse": 6.5, "mae": 4.1, "r2": 0.82, "slope": 0.67, "icpt": 27.1},
    "xgb_default": {"rmse": 13.1, "mae": 10.5, "r2": 0.68},
    "rf_default": {"rmse": 11.9, "mae": 9.3, "r2": 0.60},
    "r2_surface_energy": 0.40,
}


def load_s4(path="data/sena_s4.csv"):
    """表 S4（26行 × F_1〜F_15・WCA）。行の名前を略号にする。"""
    df = pd.read_csv(path)
    need = ["F_1"] + MODEL1 + ["WCA"]
    miss = [c for c in need if c not in df.columns]
    if miss:
        raise SystemExit(f"{path} に列 {miss} がありません")
    if len(df) != 26:
        raise SystemExit(f"{path} は26行のはずですが {len(df)} 行です")
    df["abbr"] = [ABBR[int(i)] for i in df["F_1"]]
    return df.set_index("abbr")


def metrics(y, yhat):
    """論文の式1〜3（RMSE・MAE・R²＝1−残差平方和/全平方和）と、図3 の回帰直線。"""
    y = np.asarray(y, float)
    yhat = np.asarray(yhat, float)
    e = yhat - y
    rmse = float(np.sqrt(np.mean(e ** 2)))
    mae = float(np.mean(np.abs(e)))
    r2 = float(1 - np.sum(e ** 2) / np.sum((y - y.mean()) ** 2))
    slope, icpt = np.polyfit(y, yhat, 1)          # 予測 = slope × 実験 + icpt（図3 の線）
    r2_line = float(np.corrcoef(y, yhat)[0, 1] ** 2)  # 直線の当てはまり（相関の2乗）
    return {"rmse": rmse, "mae": mae, "r2": r2, "slope": float(slope),
            "icpt": float(icpt), "r2_line": r2_line}


def fmt(m):
    return (f"RMSE {m['rmse']:5.2f}°  MAE {m['mae']:5.2f}°  R² {m['r2']:5.2f}  "
            f"傾き {m['slope']:4.2f}  (直線の R² {m['r2_line']:4.2f})")


def make_xgb(n_estimators=100, max_depth=6, learning_rate=0.3, seed=0):
    from xgboost import XGBRegressor
    return XGBRegressor(n_estimators=int(n_estimators), max_depth=int(max_depth),
                        learning_rate=float(learning_rate), random_state=seed,
                        n_jobs=1, tree_method="exact")


class MinMaxWrap:
    """論文と同じ前処理：学習データだけで X と y を 0〜1 にしてから学習し、予測を元に戻す。"""

    def __init__(self, model):
        self.model = model

    def fit(self, X, y):
        X = np.asarray(X, float)
        y = np.asarray(y, float)
        self.xmin, self.xmax = X.min(0), X.max(0)
        self.xr = np.where(self.xmax > self.xmin, self.xmax - self.xmin, 1.0)
        self.ymin, self.yr = y.min(), (y.max() - y.min())
        self.model.fit((X - self.xmin) / self.xr, (y - self.ymin) / self.yr)
        return self

    def predict(self, X):
        Xs = (np.asarray(X, float) - self.xmin) / self.xr
        return self.model.predict(Xs) * self.yr + self.ymin


def loocv(factory, X, y, per_fold=None):
    """1つ抜き交差検証。factory(i) が i 番目を抜いたときのモデルを返す。
    戻り値：予測（26個）と、各回のモデル（重要度を見るため）。"""
    X = np.asarray(X, float)
    y = np.asarray(y, float)
    n = len(y)
    pred = np.empty(n)
    models = []
    for i in range(n):
        tr = np.arange(n) != i
        m = factory(i)
        m.fit(X[tr], y[tr])
        pred[i] = float(np.ravel(m.predict(X[i:i + 1]))[0])
        models.append(m)
    return pred, models


def kfold_cv(factory, X, y, k=5, seed=0):
    """k 分割の交差検証（分け方を seed で変える）。"""
    X = np.asarray(X, float)
    y = np.asarray(y, float)
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(y))
    pred = np.empty(len(y))
    for f in range(k):
        te = idx[f::k]
        tr = np.setdiff1d(idx, te)
        m = factory(f)
        m.fit(X[tr], y[tr])
        pred[te] = np.ravel(m.predict(X[te]))
    return pred


def gain_importance(model, names):
    """XGBoost の total gain を、合計1に正規化して返す（論文の表2・表3 と同じ指標）。"""
    booster = model.model.get_booster() if isinstance(model, MinMaxWrap) else model.get_booster()
    g = booster.get_score(importance_type="total_gain")
    v = np.array([g.get(f"f{j}", 0.0) for j in range(len(names))])
    return v / v.sum() if v.sum() > 0 else v


def neumann_theta(gamma_s, gamma_l=72.8, beta=0.0001247):
    """Neumann の状態方程式（Young の式＋界面張力の経験式）で、平らな面の接触角 [度]。
    cosθ = −1 + 2√(γs/γl)·exp(−β(γl−γs)²)、γ は mN/m（= mJ/m²）。"""
    gs = np.asarray(gamma_s, float)
    c = -1 + 2 * np.sqrt(gs / gamma_l) * np.exp(-beta * (gamma_l - gs) ** 2)
    return np.degrees(np.arccos(np.clip(c, -1, 1)))


def wenzel(theta_y_deg, r):
    """Wenzel の式 cosθ* = r cosθY（r ≥ 1 は実際の面積 / 見かけの面積）。"""
    c = np.clip(r * np.cos(np.radians(theta_y_deg)), -1, 1)
    return np.degrees(np.arccos(c))


def set_jp_font():
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib import font_manager
    have = {f.name for f in font_manager.fontManager.ttflist}
    jp = next((n for n in ["Hiragino Sans", "Noto Sans CJK JP"] if n in have), None)  # Mac／Colab
    plt.rcParams["font.family"] = [jp, "sans-serif"] if jp else ["sans-serif"]
    plt.rcParams["axes.unicode_minus"] = False
    return plt


def save_json(obj, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    def conv(o):
        if isinstance(o, (np.floating,)):
            return float(o)
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        raise TypeError(type(o))
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=1, default=conv))


In [ ]:
%%writefile si_to_csv.py
"""Sena et al. 2025 の SI（jp4c06608_si_001.pdf）の表を CSV にし、写し間違いを確かめる。

使い方：
  uv run python si_to_csv.py si/jp4c06608_si_001.pdf      # PDF から data/*.csv を作って確かめる
  uv run python si_to_csv.py --check-only                  # 手で写した data/*.csv を確かめるだけ
PDF は自分のブラウザで落とした物を使う（このスクリプトはネットにつながない）。
"""
import argparse
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from ymi_core import ABBR, PAPER, metrics

NUM = r"-?\d+(?:\.\d*)?"


def pdf_text(path):
    import pdfplumber
    with pdfplumber.open(path) as pdf:
        return [p.extract_text() or "" for p in pdf.pages]


def between(text, start, end):
    i = text.find(start)
    if i < 0:
        raise SystemExit(f"「{start}」が見つかりません（PDF の版が違う？）")
    j = text.find(end, i + len(start)) if end else -1
    return text[i:j if j > 0 else None]


def parse_s1(t):
    rows = []
    for line in between(t, "Table S1.", "Table S2.").splitlines()[2:]:
        m = re.match(rf"(.+?)\s+({NUM})\s*±\s*({NUM})\s+({NUM})$", line.strip())
        if m:
            name = m.group(1)
            ab = re.findall(r"\(([^()]+)\)", name)
            rows.append({"polymer": name, "abbr": ab[0].upper().replace("NYLON6", "NYLON6") if ab else name,
                         "wca_inhouse": float(m.group(2)), "wca_sd": float(m.group(3)),
                         "sa_nm": float(m.group(4))})
    return pd.DataFrame(rows)


def parse_s3(t):
    rows = []
    for line in between(t, "Table S3.", "Table S4.").splitlines():
        m = re.match(rf"(.+\))\s+({NUM})$", line.strip())
        if m:
            rows.append({"polymer": m.group(1), "dg_solv": float(m.group(2))})
    df = pd.DataFrame(rows)
    df.insert(0, "F_1", range(len(df)))
    df.insert(1, "abbr", ABBR[:len(df)])
    return df


def parse_s4(t):
    rows = []
    for line in between(t, "Table S4.", "Table S5.").splitlines():
        parts = line.split()
        if len(parts) == 16 and all(re.fullmatch(NUM, p) for p in parts):
            rows.append([float(p) for p in parts])
    cols = ["F_1"] + [f"F_{i}" for i in range(2, 16)] + ["WCA"]
    df = pd.DataFrame(rows, columns=cols)
    df["F_1"] = df["F_1"].astype(int)
    return df


def parse_s6(t):
    block = between(t, "Table S6.", "Table S7.")
    m1, m2 = block.split("Model 2", 1)
    out = []
    for model, part in (("model1", m1), ("model2", m2)):
        for line in part.splitlines():
            m = re.match(rf"([A-Z0-9]+)\s+(\d+)\s+(\d+)\s+(\d+)\s+({NUM})$", line.strip())
            if m:
                out.append({"model": model, "abbr": m.group(1), "F_1": int(m.group(2)),
                            "n_estimators": int(m.group(3)), "max_depth": int(m.group(4)),
                            "learning_rate": float(m.group(5))})
    return pd.DataFrame(out)


def parse_pred(t, start, end):
    rows = []
    for line in between(t, start, end).splitlines():
        m = re.match(rf"([A-Z0-9]+)\s+(\d+)\s+({NUM})\s+({NUM})\s+({NUM})$", line.strip())
        if m:
            rows.append({"abbr": m.group(1), "F_1": int(m.group(2)), "exp": float(m.group(3)),
                         "pred": float(m.group(4)), "diff": float(m.group(5))})
    return pd.DataFrame(rows)


def check(d):
    """写し間違いを、表どうしの関係と論文の表4 で確かめる。問題の数を返す。"""
    bad = 0
    s4, s1, s3 = d["s4"], d["s1"], d["s3"]
    print("== 確かめ ==")
    ok = (len(s4) == 26) and list(s4["F_1"]) == list(range(26))
    print(f"[{'OK' if ok else 'NG'}] 表S4 は26行で、番号が 0〜25 の順")
    bad += not ok
    # 1) 表S3 の ΔGsolv と 表S4 の F_4（符号を取って整数にした値）
    diff = np.abs(np.trunc(np.abs(s3["dg_solv"].to_numpy())) - s4["F_4"].to_numpy())
    ok = bool(np.all(diff <= 1))
    print(f"[{'OK' if ok else 'NG'}] 表S3 の |ΔGsolv| と 表S4 の F_4 が1以内（最大の差 {diff.max():.0f}）")
    bad += not ok
    # 2) 表S1（自分たちで測った Sa）と 表S4 の F_3
    m = s1.merge(s4.assign(abbr=[ABBR[i] for i in s4["F_1"]]), on="abbr", how="left")
    dsa = (m["sa_nm"] - m["F_3"]).abs()
    print(f"[参考] 表S1 の Sa と 表S4 の F_3 が一致した高分子 {int((dsa < 1).sum())}/{len(m)}"
          f"（違う物：{', '.join(m.loc[dsa >= 1, 'abbr'])}）")
    # 3) 表S8・S9 の「実験値」が 表S4 の WCA と同じか、差の列が計算と合うか、表4 が再現するか
    for key, name in (("s8", "model1"), ("s9", "model2")):
        p = d[key]
        ok1 = len(p) == 26 and np.allclose(p["exp"].to_numpy(), s4["WCA"].to_numpy())
        ok2 = np.allclose(p["pred"] - p["exp"], p["diff"], atol=0.02) or \
            np.allclose(p["exp"] - p["pred"], p["diff"], atol=0.02)
        mt = metrics(p["exp"], p["pred"])
        ref = PAPER[name]
        # 表4 は小数1けたなので ±0.1 まで。R² は2つの定義のどちらかで合えばよい（7-2 で説明）
        ok3 = abs(mt["rmse"] - ref["rmse"]) <= 0.1 and abs(mt["mae"] - ref["mae"]) <= 0.1 \
            and min(abs(mt["r2"] - ref["r2"]), abs(mt["r2_line"] - ref["r2"])) < 0.006
        print(f"[{'OK' if ok1 else 'NG'}] 表{key.upper()} の実験値 = 表S4 の WCA")
        print(f"[{'OK' if ok2 else 'NG'}] 表{key.upper()} の差の列 = 予測 − 実験（か、その逆）")
        print(f"[{'OK' if ok3 else 'NG'}] 表{key.upper()} から計算した指標が論文の表4 と一致："
              f"RMSE {mt['rmse']:.2f}（論文 {ref['rmse']}）MAE {mt['mae']:.2f}（{ref['mae']}）"
              f"傾き {mt['slope']:.2f}（{ref['slope']}）")
        print(f"      R²：式3（1−残差/全体）{mt['r2']:.3f}、相関の2乗 {mt['r2_line']:.3f}（論文 {ref['r2']}）")
        bad += (not ok1) + (not ok2) + (not ok3)
    # 4) 表S6：26行ずつ、番号の順
    for name in ("model1", "model2"):
        h = d["s6"][d["s6"]["model"] == name]
        ok = len(h) == 26 and list(h["F_1"]) == list(range(26))
        print(f"[{'OK' if ok else 'NG'}] 表S6 の {name} は26行")
        bad += not ok
    print("問題なし" if bad == 0 else f"問題 {bad} 件：PDF と見比べて直してください")
    return bad


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("pdf", nargs="?")
    ap.add_argument("--out", default="data")
    ap.add_argument("--check-only", action="store_true")
    a = ap.parse_args()
    out = Path(a.out)
    names = {"s1": "sena_s1.csv", "s3": "sena_s3.csv", "s4": "sena_s4.csv",
             "s6": "sena_s6.csv", "s8": "sena_s8.csv", "s9": "sena_s9.csv"}
    if a.check_only:
        d = {k: pd.read_csv(out / v) for k, v in names.items()}
        sys.exit(1 if check(d) else 0)
    if not a.pdf or not Path(a.pdf).exists():
        raise SystemExit("SI の PDF のパスを指定してください（例：si/jp4c06608_si_001.pdf）")
    t = "\n".join(pdf_text(a.pdf))
    d = {"s1": parse_s1(t), "s3": parse_s3(t), "s4": parse_s4(t), "s6": parse_s6(t),
         "s8": parse_pred(t, "Table S8.", "Table S9."),
         "s9": parse_pred(t, "Table S9.", "Table S10.")}
    out.mkdir(exist_ok=True)
    for k, v in names.items():
        d[k].to_csv(out / v, index=False)
        print(f"{out / v}：{len(d[k])} 行")
    sys.exit(1 if check(d) else 0)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile features_ymi.py
"""くり返し単位の SMILES から、論文の F_5〜F_15（RDKit の記述子）を計算し、表 S4 と比べる。

  uv run python features_ymi.py                       # 三量体（端は H）で計算し、表S4 と比べる
  uv run python features_ymi.py --n 1 --cap H         # 単量体（くり返し単位1つ）
  uv run python features_ymi.py --n 3 --cap CH3       # 三量体の端をメチル基でふさぐ
  uv run python features_ymi.py --scan                # n=1・3 × 端 H・CH3 の4通りで一致の数を比べる
出力：data/rdkit_n<n>_<cap>.csv（列 F_1, abbr, smiles, F_5〜F_15）
"""
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import Crippen, Descriptors, Lipinski, rdMolDescriptors

RDLogger.DisableLog("rdApp.*")
COLS = [f"F_{i}" for i in range(5, 16)]
TOL = {"F_7": 0.02, "F_10": 0.05, "F_12": 1.0, "F_13": 1.0}   # 小数の列の許し（ほかは整数で一致）


def oligomer(repeat_smiles, n=3, cap="H"):
    """くり返し単位（つなぎ目を * で2か所）を n 個つなぎ、両端を H か CH3 でふさいだ分子を返す。"""
    unit = Chem.MolFromSmiles(repeat_smiles)
    if unit is None:
        raise ValueError(f"SMILES が読めません：{repeat_smiles}")
    stars = [a.GetIdx() for a in unit.GetAtoms() if a.GetAtomicNum() == 0]
    if len(stars) != 2:
        raise ValueError(f"* は2つ必要です（{len(stars)} 個）：{repeat_smiles}")
    head_star, tail_star = stars
    head = unit.GetAtomWithIdx(head_star).GetNeighbors()[0].GetIdx()
    tail = unit.GetAtomWithIdx(tail_star).GetNeighbors()[0].GetIdx()
    mol = Chem.RWMol()
    heads, tails = [], []
    for _ in range(n):
        offset = mol.GetNumAtoms()
        mp = {}
        for a in unit.GetAtoms():
            if a.GetAtomicNum() == 0:
                continue
            na = Chem.Atom(a.GetAtomicNum())
            na.SetFormalCharge(a.GetFormalCharge())
            na.SetIsAromatic(a.GetIsAromatic())
            mp[a.GetIdx()] = mol.AddAtom(na)
        for b in unit.GetBonds():
            i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
            if i in mp and j in mp:
                mol.AddBond(mp[i], mp[j], b.GetBondType())
        heads.append(mp[head])
        tails.append(mp[tail])
    for k in range(n - 1):                      # k 番目の尾と k+1 番目の頭をつなぐ
        mol.AddBond(tails[k], heads[k + 1], Chem.BondType.SINGLE)
    if cap == "CH3":
        for end in (heads[0], tails[-1]):
            c = mol.AddAtom(Chem.Atom(6))
            mol.AddBond(end, c, Chem.BondType.SINGLE)
    m = mol.GetMol()
    for a in m.GetAtoms():
        a.SetNoImplicit(False)
        a.SetNumExplicitHs(0)
    Chem.SanitizeMol(m)
    return m


def descriptors(m):
    """論文の表2 の F_5〜F_15 に当たる記述子（どの関数を使ったかは論文に書かれていないので、この対応は推測）。"""
    mh = Chem.AddHs(m)
    hyb = [a.GetHybridization() for a in m.GetAtoms()]
    return {
        "F_5": sum(h == Chem.HybridizationType.SP3 for h in hyb),
        "F_6": sum(h == Chem.HybridizationType.SP2 for h in hyb),
        "F_7": round(rdMolDescriptors.CalcFractionCSP3(m), 2),
        "F_8": Lipinski.NumHAcceptors(m),
        "F_9": Lipinski.NumHDonors(m),
        "F_10": round(Crippen.MolLogP(m), 2),
        "F_11": m.GetNumHeavyAtoms(),
        "F_12": round(Descriptors.MolWt(m), 1),
        "F_13": round(Descriptors.HeavyAtomMolWt(m), 1),
        "F_14": sum(a.GetAtomicNum() == 1 for a in mh.GetAtoms()),
        "F_15": Descriptors.NumValenceElectrons(m),
    }


def compute(poly, n, cap):
    rows = []
    for _, r in poly.iterrows():
        m = oligomer(r["repeat_smiles"], n, cap)
        rows.append({"F_1": r["F_1"], "abbr": r["abbr"], "smiles": Chem.MolToSmiles(m), **descriptors(m)})
    return pd.DataFrame(rows)


def agree(ours, s4):
    """列ごと・高分子ごとに、表S4 と一致したかを True/False の表で返す。"""
    a = ours.set_index("F_1")[COLS]
    b = s4.set_index("F_1")[COLS]
    ok = pd.DataFrame(index=a.index, columns=COLS, dtype=bool)
    for c in COLS:
        ok[c] = (a[c] - b[c]).abs() <= TOL.get(c, 0.5) + 1e-9   # 1e-9：0.29 − 0.27 が 0.02 より少し大きく出る浮動小数の誤差を許す
    return ok


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--poly", default="polymers.csv")
    ap.add_argument("--s4", default="data/sena_s4.csv")
    ap.add_argument("--n", type=int, default=3)
    ap.add_argument("--cap", choices=["H", "CH3"], default="H")
    ap.add_argument("--scan", action="store_true")
    a = ap.parse_args()
    poly = pd.read_csv(a.poly)
    s4 = pd.read_csv(a.s4) if Path(a.s4).exists() else None
    combos = [(1, "H"), (1, "CH3"), (3, "H"), (3, "CH3")] if a.scan else [(a.n, a.cap)]
    for n, cap in combos:
        ours = compute(poly, n, cap)
        out = Path("data") / f"rdkit_n{n}_{cap}.csv"
        out.parent.mkdir(exist_ok=True)
        ours.to_csv(out, index=False)
        print(f"== n={n}・端={cap} → {out}")
        if s4 is None:
            continue
        ok = agree(ours, s4)
        per_col = ok.sum(axis=0)
        print("列ごとの一致（26 のうち）：" + "  ".join(f"{c}:{int(v)}" for c, v in per_col.items()))
        print(f"全部の列が一致した高分子：{int(ok.all(axis=1).sum())}/26"
              f"（{', '.join(ours.set_index('F_1').loc[ok.all(axis=1), 'abbr'])}）")
        if not a.scan:
            # 分子量（F_12）が合わない物は、論文が別の構造・別の長さで計算した可能性が高い
            m = ours.set_index("F_1").merge(s4.set_index("F_1")[COLS].add_suffix("_paper"),
                                              left_index=True, right_index=True)
            bad = m[~ok["F_12"]]
            for f1, r in bad.iterrows():
                print(f"  {r['abbr']:7s} 分子量 ours {r['F_12']:7.1f} / 論文 {r['F_12_paper']:7.1f}"
                      f"  H の数 {r['F_14']:3.0f} / {r['F_14_paper']:3.0f}"
                      f"  LogP {r['F_10']:5.2f} / {r['F_10_paper']:5.2f}")
            # 論文の値と自分の値の相関（順位がそろっていれば、木のモデルへの影響は小さい）
            rho = {c: ours[c].rank().corr(s4[c].rank()) for c in COLS}
            print("順位相関：" + "  ".join(f"{c}:{v:.2f}" for c, v in rho.items()))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile reproduce_ymi.py
"""Sena et al. 2025 の表4・表S5・図3・表2/表3・図S7 を再現する（主解析）。

  uv run python reproduce_ymi.py                    # 論文の表S4 の特徴量で
  uv run python reproduce_ymi.py --rdkit data/rdkit_n3_H.csv   # F_5〜F_15 を自分の RDKit の値に入れ替える
  uv run python reproduce_ymi.py --data synth --out out_synth  # 合成の代わりのデータで練習
出力：<out>/reproduce.json、<out>/fig3_repro.png、<out>/importance.png
"""
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import warnings

from sklearn.exceptions import ConvergenceWarning

from ymi_core import (ABBR, FEAT, MODEL1, MODEL2, PAPER, MinMaxWrap, fmt, gain_importance,
                      load_s4, loocv, make_xgb, metrics, save_json, set_jp_font)


def make_gp():
    """特徴量を標準化 → GP（y も標準化）。雑音の分散の下限 0.02 は、元の単位で SD 約2°
    （表S1 の測定の ±1〜6° より小さい雑音は考えない）。下限に張り付いた回数は extend_ymi.py が数える。"""
    k = ConstantKernel(1.0, (1e-2, 1e2)) * Matern(length_scale=3.0, length_scale_bounds=(0.3, 100.0), nu=2.5) \
        + WhiteKernel(0.3, (0.02, 10.0))
    return make_pipeline(StandardScaler(),
                         GaussianProcessRegressor(k, normalize_y=True, n_restarts_optimizer=3, random_state=0))


def main():
    warnings.filterwarnings("ignore", category=ConvergenceWarning)   # GP の境界の警告（extend_ymi.py で数える）
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data")
    ap.add_argument("--rdkit", default=None, help="F_5〜F_15 を入れ替える CSV（features_ymi.py の出力）")
    ap.add_argument("--out", default="out")
    a = ap.parse_args()
    D, OUT = Path(a.data), Path(a.out)
    OUT.mkdir(exist_ok=True)
    s4 = load_s4(D / "sena_s4.csv")
    if a.rdkit:
        rd = pd.read_csv(a.rdkit).set_index("abbr")
        for c in [f"F_{i}" for i in range(5, 16)]:
            s4[c] = rd.loc[s4.index, c].to_numpy()
        print(f"F_5〜F_15 を {a.rdkit} の値に入れ替えました")
    y = s4["WCA"].to_numpy()
    res = {"n": len(y), "sd_wca": float(y.std(ddof=1))}
    print(f"高分子 {len(y)} 種、WCA の平均 {y.mean():.1f}°・SD {y.std(ddof=1):.1f}°")

    # A. 論文の表S8・S9 から、論文の表4 を計算し直す（数字の出どころの確かめ）
    for key, name in (("s8", "model1"), ("s9", "model2")):
        f = D / f"sena_{key}.csv"
        if f.exists():
            p = pd.read_csv(f)
            m = metrics(p["exp"], p["pred"])
            res[f"paper_{name}_from_table"] = m
            print(f"[A] 論文の表{key.upper()} → {fmt(m)}（論文の表4：RMSE {PAPER[name]['rmse']}・"
                  f"MAE {PAPER[name]['mae']}・R² {PAPER[name]['r2']}）")

    # B. 既定のハイパーパラメータの XGBoost（論文の表S5 と比べる）
    for name, cols in (("model1", MODEL1), ("model2", MODEL2)):
        X = s4[cols].to_numpy()
        p, _ = loocv(lambda i: MinMaxWrap(make_xgb()), X, y)
        m = metrics(y, p)
        res[f"default_{name}"] = m
        print(f"[B] XGBoost 既定値 {name}：{fmt(m)}")
    print(f"    （論文の表S5 の XGBoost：RMSE {PAPER['xgb_default']['rmse']}・MAE {PAPER['xgb_default']['mae']}・"
          f"R² {PAPER['xgb_default']['r2']}。どの特徴量の組かは書かれていない）")

    # C. 表S6 の「1つ抜くごとのハイパーパラメータ」で LOOCV（論文の手順そのもの）
    s6f = D / "sena_s6.csv"
    preds = {}
    imp = {}
    if s6f.exists():
        s6 = pd.read_csv(s6f)
        for name, cols, key in (("model1", MODEL1, "s8"), ("model2", MODEL2, "s9")):
            h = s6[s6["model"] == name].set_index("F_1")
            X = s4[cols].to_numpy()
            fac = lambda i, h=h: MinMaxWrap(make_xgb(h.loc[i, "n_estimators"], h.loc[i, "max_depth"],
                                                     h.loc[i, "learning_rate"]))
            p, models = loocv(fac, X, y)
            preds[name] = p
            m = metrics(y, p)
            res[f"s6_{name}"] = m
            ref = pd.read_csv(D / f"sena_{key}.csv")["pred"].to_numpy()
            d = p - ref
            res[f"s6_{name}_vs_paper"] = {"rms_diff": float(np.sqrt(np.mean(d ** 2))),
                                         "max_abs_diff": float(np.abs(d).max()),
                                         "worst": ABBR[int(np.abs(d).argmax())],
                                         "corr": float(np.corrcoef(p, ref)[0, 1])}
            print(f"[C] 表S6 のハイパーパラメータ {name}：{fmt(m)}")
            print(f"    論文の予測（表{key.upper()}）との差：RMS {res[f's6_{name}_vs_paper']['rms_diff']:.2f}°、"
                  f"最大 {np.abs(d).max():.2f}°（{ABBR[int(np.abs(d).argmax())]}）、相関 {np.corrcoef(p, ref)[0, 1]:.4f}")
            g = np.mean([gain_importance(mm, cols) for mm in models], axis=0)
            imp[name] = dict(zip(cols, g / g.sum()))
            top = sorted(imp[name].items(), key=lambda kv: -kv[1])[:5]
            print("    重要度（26回の平均の total gain）上位5："
                  + "、".join(f"{c}（{FEAT[c]}）{v:.3f}" for c, v in top))
        res["importance"] = imp
    else:
        print("[C] sena_s6.csv がないので、表S6 を使う再現はとばします（合成データのとき）")

    # D. 比べる相手（ベースライン）：平均だけ、表面エネルギーだけの直線、RandomForest、ガウス過程
    X1 = s4[MODEL1].to_numpy()
    p_mean = np.array([np.delete(y, i).mean() for i in range(len(y))])
    res["mean_only"] = metrics(y, p_mean)
    print(f"[D] 平均だけ（学習の平均を答える）：{fmt(res['mean_only'])}")
    r = stats.linregress(s4["F_2"], y)
    res["se_in_sample_r2"] = float(r.rvalue ** 2)
    print(f"[D] 表面エネルギーだけの直線（全データ、図S7）：R² {r.rvalue ** 2:.3f}"
          f"（論文 {PAPER['r2_surface_energy']}）、傾き {r.slope:.2f}°/(mJ/m²)")
    from sklearn.linear_model import LinearRegression
    p_lin, _ = loocv(lambda i: LinearRegression(), s4[["F_2"]].to_numpy(), y)
    res["se_loocv"] = metrics(y, p_lin)
    print(f"[D] 表面エネルギーだけの直線（LOOCV）：{fmt(res['se_loocv'])}")
    p_rf, _ = loocv(lambda i: RandomForestRegressor(n_estimators=500, random_state=0, n_jobs=1), X1, y)
    res["rf_model1"] = metrics(y, p_rf)
    print(f"[D] RandomForest（500本）model1：{fmt(res['rf_model1'])}")
    p_rf2, _ = loocv(lambda i: RandomForestRegressor(n_estimators=500, random_state=0, n_jobs=1),
                     s4[MODEL2].to_numpy(), y)
    res["rf_model2"] = metrics(y, p_rf2)
    dd = np.abs(p_mean - y) - np.abs(p_rf2 - y)
    res["rf_model2_vs_mean_p"] = float(stats.wilcoxon(dd).pvalue)
    print(f"[D] RandomForest（500本、設定を調整しない）model2：{fmt(res['rf_model2'])}")
    print(f"    平均だけとの対応のある比較（ウィルコクソン）p = {res['rf_model2_vs_mean_p']:.3f}"
          "（26種では検出力が低い。区別できない ≠ 役に立たない）")
    p_gp, _ = loocv(lambda i: make_gp(), X1, y)
    res["gp_model1"] = metrics(y, p_gp)
    print(f"[D] ガウス過程（Matérn 5/2、標準化）model1：{fmt(res['gp_model1'])}")
    preds.update({"mean_only": p_mean, "rf_model1": p_rf, "gp_model1": p_gp})

    save_json({"results": res, "pred": {k: v for k, v in preds.items()}, "abbr": list(s4.index)},
              OUT / "reproduce.json")

    # 図：予測と実験（図3 と同じ形）と重要度
    plt = set_jp_font()
    fig, ax = plt.subplots(figsize=(5.2, 5))
    lim = (40, 120)
    ax.plot(lim, lim, color="0.6", lw=1)
    for name, mk, col in (("model1", "o", "C3"), ("model2", "s", "C0")):
        if name in preds:
            ax.scatter(y, preds[name], marker=mk, color=col, s=28, label=f"{name}（表S6 の設定）")
    ax.scatter(y, p_gp, marker="^", facecolor="none", edgecolor="0.3", s=30, label="ガウス過程 model1")
    ax.set(xlim=lim, ylim=lim, xlabel="実験の WCA（表S4）[°]", ylabel="LOOCV の予測 [°]")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "fig3_repro.png", dpi=150)
    if imp:
        fig, ax = plt.subplots(figsize=(6, 4))
        cols = MODEL1
        v = [imp["model1"].get(c, 0) for c in cols]
        ax.barh([f"{c} {FEAT[c]}" for c in cols][::-1], v[::-1], color="C3")
        ax.set_xlabel("total gain（合計1）")
        fig.tight_layout()
        fig.savefig(OUT / "importance.png", dpi=150)
    print(f"→ {OUT}/reproduce.json・fig3_repro.png・importance.png")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile deviation_ymi.py
"""ずれの分析：データ漏れ（ハイパーパラメータの選び方）・入れ子の交差検証・CV のばらつき・
重要度の不確かさ・多重比較・別の出どころの値との食い違い。

  uv run python deviation_ymi.py                  # すべて（入れ子の CV に数分かかる）
  uv run python deviation_ymi.py --skip-nested    # 入れ子の CV をとばす（数十秒）
  uv run python deviation_ymi.py --full-grid      # 入れ子の CV も 252 通りの格子で（10分ほど）
  uv run python deviation_ymi.py --data synth --out out_synth
出力：<out>/deviation.json、<out>/cv_spread.png、<out>/rank_stability.png
"""
import argparse
import itertools
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

from ymi_core import (ABBR, FEAT, MODEL1, MODEL2, MinMaxWrap, gain_importance, kfold_cv,
                      load_s4, make_xgb, metrics, save_json, set_jp_font)

# 表S6 に出てくる値を含む格子（6 × 6 × 7 = 252 通り）と、入れ子の CV 用の小さい格子（4 × 4 × 4 = 64 通り）
FULL = list(itertools.product([10, 15, 20, 50, 60, 100], [1, 2, 3, 4, 5, 6],
                              [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]))
SMALL = list(itertools.product([10, 20, 50, 100], [1, 2, 3, 6], [0.1, 0.3, 0.5, 0.7]))


def heldout_errors(X, y, grid):
    """E[i, g]：i 番目を抜いて格子の g 番目の設定で学習したときの、i 番目の予測の誤差。"""
    n = len(y)
    E = np.zeros((n, len(grid)))
    for i in range(n):
        tr = np.arange(n) != i
        for g, (ne, d, lr) in enumerate(grid):
            m = MinMaxWrap(make_xgb(ne, d, lr)).fit(X[tr], y[tr])
            E[i, g] = m.predict(X[i:i + 1])[0] - y[i]
    return E


def nested_loocv(X, y, grid, inner="loo"):
    """入れ子の交差検証：外側で1つ抜き、内側（残り25個だけ）で設定を選び、外した点で1回だけ評価する。"""
    n = len(y)
    pred = np.empty(n)
    chosen = []
    for i in range(n):
        idx = np.flatnonzero(np.arange(n) != i)
        Xi, yi = X[idx], y[idx]
        if inner == "loo":
            Ein = heldout_errors(Xi, yi, grid)
            score = (Ein ** 2).mean(0)
        else:
            k = int(inner)
            score = np.zeros(len(grid))
            folds = np.array_split(np.random.default_rng(i).permutation(len(yi)), k)
            for g, (ne, d, lr) in enumerate(grid):
                se = 0.0
                for te in folds:
                    tr = np.setdiff1d(np.arange(len(yi)), te)
                    m = MinMaxWrap(make_xgb(ne, d, lr)).fit(Xi[tr], yi[tr])
                    se += np.sum((m.predict(Xi[te]) - yi[te]) ** 2)
                score[g] = se / len(yi)
        gb = int(np.argmin(score))
        chosen.append(grid[gb])
        pred[i] = MinMaxWrap(make_xgb(*grid[gb])).fit(Xi, yi).predict(X[i:i + 1])[0]
    return pred, chosen


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data")
    ap.add_argument("--out", default="out")
    ap.add_argument("--skip-nested", action="store_true")
    ap.add_argument("--full-grid", action="store_true")
    ap.add_argument("--inner", default="loo", help="内側の CV：loo か 分割数（例 5）")
    ap.add_argument("--seeds", type=int, default=50)
    ap.add_argument("--boot", type=int, default=200)
    a = ap.parse_args()
    D, OUT = Path(a.data), Path(a.out)
    OUT.mkdir(exist_ok=True)
    s4 = load_s4(D / "sena_s4.csv")
    y = s4["WCA"].to_numpy()
    n = len(y)
    res = {}
    rng = np.random.default_rng(0)

    # 1. 表S6 の設定は、外した点の誤差で見てどのくらい「当たり」か（データ漏れの手がかり）
    s6f = D / "sena_s6.csv"
    Eall = {}
    for name, cols in (("model1", MODEL1), ("model2", MODEL2)):
        X = s4[cols].to_numpy()
        t0 = time.time()
        E = heldout_errors(X, y, FULL)
        Eall[name] = np.abs(E)
        oracle = np.abs(E).min(1)
        r = {"oracle_rmse": float(np.sqrt(np.mean(oracle ** 2))), "oracle_mae": float(oracle.mean())}
        rm = np.sqrt((E ** 2).mean(0))
        gb = int(rm.argmin())
        r["best_fixed"] = {"setting": FULL[gb], "rmse": float(rm[gb])}
        print(f"[1] {name}：格子 {len(FULL)} 通り × 26 回（{time.time() - t0:.0f}秒）")
        print(f"    外した点ごとに一番よい設定を選ぶ（漏れのある評価の上限の目安）：RMSE {r['oracle_rmse']:.2f}°・"
              f"MAE {r['oracle_mae']:.2f}°")
        print(f"    1つの設定を26回の LOOCV で選ぶ（これも少し楽観的）：{FULL[gb]} → RMSE {rm[gb]:.2f}°")
        # 比べる相手1：1つに固定した設定（外した点を見ていない）なら、「順位が0.5未満」の数はいくつになるか
        A = np.abs(E)
        below = np.array([int(((A < A[:, [g]]).mean(1) < 0.5).sum()) for g in range(len(FULL))])
        r["fixed_below_half"] = {"max": int(below.max()), "best_setting": int(below[gb]),
                                 "median": float(np.median(below))}
        print(f"    比べる相手：1つに固定した設定 {len(FULL)} 通りで、順位0.5未満の高分子の数は中央値 "
              f"{np.median(below):.0f}・最大 {below.max()}/{n}（LOOCV で最良の設定 {FULL[gb]} は {below[gb]}/{n}）")
        if s6f.exists():
            h = pd.read_csv(s6f)
            h = h[h["model"] == name].set_index("F_1")
            pct = []
            for i in range(n):
                g = FULL.index((int(h.loc[i, "n_estimators"]), int(h.loc[i, "max_depth"]),
                                float(h.loc[i, "learning_rate"])))
                pct.append(float((np.abs(E[i]) < np.abs(E[i, g])).mean()))
            pct = np.array(pct)
            k = int((pct < 0.5).sum())
            p_sign = stats.binomtest(k, n, 0.5, alternative="greater").pvalue
            r["s6_percentile"] = {"median": float(np.median(pct)), "below_half": k, "p_sign": float(p_sign),
                                  "values": pct.round(3).tolist()}
            print(f"    表S6 の設定の「外した点の誤差」の順位（0＝一番よい、1＝一番悪い）：中央値 {np.median(pct):.3f}")
            print(f"    0.5未満は {k}/{n}（一様分布を仮定した符号検定の片側 p = {p_sign:.1e}）。"
                  f"固定した設定の最大 {below.max()}/{n} と比べる")
        res[name] = r

    # 2. 入れ子の交差検証（偏りのない見積もり）
    if not a.skip_nested:
        grid = FULL if a.full_grid else SMALL
        for name, cols in (("model1", MODEL1), ("model2", MODEL2)):
            t0 = time.time()
            p, chosen = nested_loocv(s4[cols].to_numpy(), y, grid, a.inner)
            m = metrics(y, p)
            res[name]["nested"] = m
            res[name]["nested_pred"] = p
            res[name]["nested_chosen"] = [list(c) for c in chosen]
            print(f"[2] 入れ子の LOOCV {name}（格子 {len(grid)} 通り、内側 {a.inner}、{time.time() - t0:.0f}秒）："
                  f"RMSE {m['rmse']:.2f}°・MAE {m['mae']:.2f}°・R² {m['r2']:.2f}（相関の2乗 {m['r2_line']:.2f}）")
            # 比べる相手2：内側（残り25種）だけで選んだ設定は、外した点の誤差で見て何位か（[1] と同じ物差し）
            A = Eall[name]
            pc = np.array([(A[i] < A[i, FULL.index(tuple(c))]).mean() for i, c in enumerate(chosen)])
            res[name]["nested_percentile"] = {"median": float(np.median(pc)), "below_half": int((pc < 0.5).sum())}
            print(f"    内側だけで選んだ設定の「外した点の誤差」の順位：中央値 {np.median(pc):.2f}、"
                  f"0.5未満 {int((pc < 0.5).sum())}/{n}（表S6 の設定と比べる）")
            # 「平均だけ」より良いか：同じ高分子どうしの |誤差| の差（対応のある比較）
            p0 = np.array([np.delete(y, i).mean() for i in range(n)])
            dd = np.abs(p0 - y) - np.abs(p - y)
            w = stats.wilcoxon(dd)
            res[name]["nested_vs_mean"] = {"mean_gain": float(dd.mean()), "wins": int((dd > 0).sum()),
                                           "p_wilcoxon": float(w.pvalue)}
            print(f"    平均だけ（MAE {np.abs(p0 - y).mean():.2f}°）より |誤差| が平均 {dd.mean():.2f}° 小さい"
                  f"（{int((dd > 0).sum())}/{n} 種で勝ち、ウィルコクソン p = {w.pvalue:.3f}。"
                  "LOOCV の誤差どうしは独立でないので目安）")

    # 3. 交差検証そのもののばらつき：同じモデルでも、分け方で RMSE が変わる
    X1 = s4[MODEL1].to_numpy()
    fixed = (50, 2, 0.1)
    rm5 = []
    for sd in range(a.seeds):
        p = kfold_cv(lambda f: MinMaxWrap(make_xgb(*fixed)), X1, y, k=5, seed=sd)
        rm5.append(metrics(y, p)["rmse"])
    rm5 = np.array(rm5)
    from ymi_core import loocv
    p_loo, _ = loocv(lambda i: MinMaxWrap(make_xgb(*fixed)), X1, y)
    e = p_loo - y
    bs = np.array([np.sqrt(np.mean(rng.choice(e, n) ** 2)) for _ in range(2000)])
    res["cv_spread"] = {"setting": fixed, "kfold5_rmse": rm5.tolist(),
                        "kfold5_min": float(rm5.min()), "kfold5_max": float(rm5.max()),
                        "loocv_rmse": float(np.sqrt(np.mean(e ** 2))),
                        "loocv_rmse_boot95": [float(np.quantile(bs, 0.025)), float(np.quantile(bs, 0.975))]}
    print(f"[3] 設定 {fixed} の model1：5分割 CV の RMSE は分け方 {a.seeds} 通りで "
          f"{rm5.min():.2f}〜{rm5.max():.2f}°（中央値 {np.median(rm5):.2f}°）")
    print(f"    LOOCV の RMSE {np.sqrt(np.mean(e ** 2)):.2f}°、26個の誤差のブートストラップ95%区間 "
          f"[{np.quantile(bs, 0.025):.2f}, {np.quantile(bs, 0.975):.2f}]°")

    # 4. 重要度の順位の不確かさ：ハイパーパラメータを変える／高分子をブートストラップする
    def ranks(model):
        g = gain_importance(model, MODEL1)
        return (-g).argsort().argsort() + 1          # 1 = いちばん重要
    R_hp = np.array([ranks(MinMaxWrap(make_xgb(*s)).fit(X1, y)) for s in SMALL])
    setting = (44, 2, 0.3)                            # 表S7 の Model 1 の平均
    R_bs = []
    for _ in range(a.boot):
        b = rng.choice(n, n)
        R_bs.append(ranks(MinMaxWrap(make_xgb(*setting)).fit(X1[b], y[b])))
    R_bs = np.array(R_bs)
    rs = {}
    for j, c in enumerate(MODEL1):
        rs[c] = {"top1_hp": float((R_hp[:, j] == 1).mean()), "top3_hp": float((R_hp[:, j] <= 3).mean()),
                 "top1_boot": float((R_bs[:, j] == 1).mean()), "top3_boot": float((R_bs[:, j] <= 3).mean()),
                 "rank_boot_95": [int(np.quantile(R_bs[:, j], 0.025)), int(np.quantile(R_bs[:, j], 0.975))]}
    res["rank_stability"] = rs
    print(f"[4] 重要度の1位になった割合（ハイパーパラメータ {len(SMALL)} 通り／ブートストラップ {a.boot} 回、設定 {setting}）")
    for c in sorted(MODEL1, key=lambda c: -rs[c]["top3_boot"])[:6]:
        print(f"    {c:5s}{FEAT[c]:12s} 1位 {rs[c]['top1_hp']:.2f} / {rs[c]['top1_boot']:.2f}、"
              f"3位以内 {rs[c]['top3_hp']:.2f} / {rs[c]['top3_boot']:.2f}、順位の95%範囲 {rs[c]['rank_boot_95']}")

    # 5. 多重比較：14本の単回帰（図S7）でいちばん良い R² は、無関係な特徴でもどのくらい出るか
    r2 = {c: float(np.corrcoef(s4[c], y)[0, 1] ** 2) for c in MODEL1}
    best = max(r2, key=r2.get)
    Z = (X1 - X1.mean(0)) / X1.std(0)
    null_max = []
    for _ in range(5000):
        yp = rng.permutation(y)
        yz = (yp - yp.mean()) / yp.std()
        null_max.append(float(np.max((Z.T @ yz / n) ** 2)))
    null_max = np.array(null_max)
    p_single = stats.pearsonr(s4[best], y).pvalue
    p_max = float((null_max >= r2[best]).mean())
    pv = np.array([stats.pearsonr(s4[c], y).pvalue for c in MODEL1])
    order = np.argsort(pv)
    holm = np.empty_like(pv)
    run = 0.0
    for k, j in enumerate(order):
        run = max(run, min(1.0, (len(pv) - k) * pv[j]))
        holm[j] = run
    res["multiple"] = {"r2": r2, "best": best, "p_single": float(p_single), "p_maxT": p_max,
                       "null_max_r2_95": float(np.quantile(null_max, 0.95)),
                       "null_max_r2_median": float(np.median(null_max)),
                       "holm": dict(zip(MODEL1, holm.round(4).tolist()))}
    print(f"[5] 単回帰の R² の最大は {best}（{FEAT[best]}）の {r2[best]:.3f}。そのままの p = {p_single:.4f}")
    print(f"    y を並べ替えたとき14本の最大 R² の中央値 {np.median(null_max):.3f}、95%点 "
          f"{np.quantile(null_max, 0.95):.3f} → 最大を選んだことを考えた p = {p_max:.4f}")
    print("    Holm で調整した p が0.05未満の特徴：" +
          ("、".join(c for c in MODEL1 if holm[MODEL1.index(c)] < 0.05) or "なし"))

    # 6. 別の出どころの値との食い違い：自分たちで測った値（表S1）と、学習に使った値（表S4、文献の平均）
    s1f = D / "sena_s1.csv"
    if s1f.exists():
        s1 = pd.read_csv(s1f).set_index("abbr")
        d = (s1["wca_inhouse"] - s4.loc[s1.index, "WCA"]).to_numpy()
        res["label_gap"] = {"n": int(len(d)), "rmse": float(np.sqrt(np.mean(d ** 2))), "mae": float(np.abs(d).mean()),
                            "mean": float(d.mean()), "sd": float(d.std(ddof=1)), "max": float(np.abs(d).max()),
                            "worst": list(s1.index[np.argsort(-np.abs(d))[:3]])}
        print(f"[6] 表S1（自分たちで新しく測った値）と表S4（文献の平均）の差：{len(d)}種で RMSE {res['label_gap']['rmse']:.2f}°、"
              f"平均 {d.mean():+.2f}°・SD {d.std(ddof=1):.2f}°、最大 {np.abs(d).max():.0f}°（{', '.join(res['label_gap']['worst'])}）")
        print("    これは表S4 そのものの雑音ではなく、2つの出どころの食い違い（新しく測った値を予測するときの目安）")

    save_json(res, OUT / "deviation.json")
    plt = set_jp_font()
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
    ax[0].hist(rm5, bins=15, color="C0")
    ax[0].axvline(res["cv_spread"]["loocv_rmse"], color="C3", ls="--", label="LOOCV")
    ax[0].set(xlabel="5分割 CV の RMSE [°]", ylabel="分け方の数", title=f"設定 {fixed} の model1")
    ax[0].legend()
    top = sorted(MODEL1, key=lambda c: -rs[c]["top3_boot"])[:6]
    ax[1].bar([FEAT[c] for c in top], [rs[c]["top1_boot"] for c in top], color="C3", label="1位")
    ax[1].bar([FEAT[c] for c in top], [rs[c]["top3_boot"] - rs[c]["top1_boot"] for c in top],
              bottom=[rs[c]["top1_boot"] for c in top], color="C1", label="2・3位")
    ax[1].set(ylabel="ブートストラップでの割合", ylim=(0, 1))
    ax[1].tick_params(axis="x", labelrotation=30, labelsize=8)
    ax[1].legend()
    fig.tight_layout()
    fig.savefig(OUT / "cv_spread.png", dpi=150)
    print(f"→ {OUT}/deviation.json・cv_spread.png")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile extend_ymi.py
"""小さな拡張：ガウス過程の予測区間・物理のベースライン（Young＋Neumann・Wenzel）・
家で測った高分子を学習から外して予測と比べる（最終演習Y・第8章とつなぐ）。

  uv run python extend_ymi.py --demo                       # 家の測定の代わりに練習用の値で
  uv run python extend_ymi.py --home home/angles_home.csv  # 第8章の contact_angle.py の出力
  uv run python extend_ymi.py --data synth --out out_synth --demo
家の写真の名前：<材料>_P<片>_D<滴>.jpg（材料は PMMA・PP・PET・PS、例 PMMA_P2_D3.jpg）
出力：<out>/extend.json、<out>/gp_intervals.png、<out>/home_vs_model.png
"""
import argparse
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.exceptions import ConvergenceWarning

from ymi_core import (MODEL1, MODEL2, MinMaxWrap, load_s4, loocv, make_xgb, metrics,
                      neumann_theta, save_json, set_jp_font)
from reproduce_ymi import make_gp
from deviation_ymi import SMALL, heldout_errors

HOME = ["PMMA", "PP", "PET", "PS"]


def gp_loocv_intervals(X, y):
    """GP の LOOCV で、予測区間（新しく測った1つの値が入る範囲）と信頼区間（平均の関数が入る範囲）を出す。"""
    n = len(y)
    mu, sd_pred, sd_lat = np.empty(n), np.empty(n), np.empty(n)
    at_bound = 0
    for i in range(n):
        tr = np.arange(n) != i
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always", ConvergenceWarning)
            pipe = make_gp().fit(X[tr], y[tr])
        at_bound += any(issubclass(x.category, ConvergenceWarning) for x in w)
        gp = pipe[-1]
        m, s = pipe.predict(X[i:i + 1], return_std=True)
        noise = gp.kernel_.k2.noise_level * gp._y_train_std ** 2   # 白色雑音の分散（元の単位）
        mu[i], sd_pred[i] = m[0], s[0]
        sd_lat[i] = np.sqrt(max(s[0] ** 2 - noise, 0.0))
    if at_bound:
        print(f"    （注意）{at_bound}/{n} 回で、GP のハイパーパラメータが探す範囲の端に張り付いた")
    return mu, sd_pred, sd_lat


def read_home(path):
    """contact_angle.py の CSV から、材料 → 片ごとの平均 → 材料の平均と95%区間（片が単位）。"""
    df = pd.read_csv(path)
    col = "theta_tan" if "theta_tan" in df.columns else "theta"
    rows = []
    for _, r in df.iterrows():
        m = re.match(r"([A-Za-z]+)_P(\d+)_D(\d+)", Path(str(r["file"])).stem)
        if m and m.group(1).upper() in HOME:
            rows.append({"mat": m.group(1).upper(), "piece": int(m.group(2)), "theta": float(r[col])})
    if not rows:
        raise SystemExit(f"{path} に <材料>_P<片>_D<滴> の名前の行がありません")
    d = pd.DataFrame(rows)
    piece = d.groupby(["mat", "piece"])["theta"].mean().reset_index()
    out = {}
    for mat, g in piece.groupby("mat"):
        k = len(g)
        mean = g["theta"].mean()
        se = g["theta"].std(ddof=1) / np.sqrt(k) if k > 1 else np.nan
        half = stats.t.ppf(0.975, k - 1) * se if k > 1 else np.nan
        out[mat] = {"mean": float(mean), "lo": float(mean - half), "hi": float(mean + half),
                    "pieces": int(k), "drops": int((d["mat"] == mat).sum())}
    return out


def demo_home(path, seed=7):
    """練習用の「家の測定」（本物ではない）。片の SD 2.5°・滴の SD 2.0°（第8章のデモと同じ大きさ）。"""
    true = {"PMMA": 71.0, "PP": 97.0, "PET": 77.0, "PS": 88.0}
    rng = np.random.default_rng(seed)
    rows = []
    for mat, t in true.items():
        for p in range(1, 5):
            u = rng.normal(0, 2.5)
            for dr in range(1, 4):
                rows.append({"file": f"{mat}_P{p}_D{dr}.jpg", "theta_tan": round(t + u + rng.normal(0, 2.0), 2)})
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(path, index=False)
    return true


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data")
    ap.add_argument("--out", default="out")
    ap.add_argument("--home", default=None)
    ap.add_argument("--demo", action="store_true")
    a = ap.parse_args()
    D, OUT = Path(a.data), Path(a.out)
    OUT.mkdir(exist_ok=True)
    s4 = load_s4(D / "sena_s4.csv")
    y = s4["WCA"].to_numpy()
    X1 = s4[MODEL1].to_numpy()
    res = {}
    warnings.filterwarnings("ignore", category=ConvergenceWarning)

    # 1. ガウス過程の予測区間と信頼区間：26個のうち何個が区間に入るか
    mu, sp, sl = gp_loocv_intervals(X1, y)
    z = 1.959964
    cov_p = float(np.mean(np.abs(y - mu) <= z * sp))
    cov_l = float(np.mean(np.abs(y - mu) <= z * sl))
    res["gp"] = {**metrics(y, mu), "cover_pred95": cov_p, "cover_latent95": cov_l,
                 "mean_halfwidth_pred": float(z * sp.mean()), "mean_halfwidth_latent": float(z * sl.mean())}
    k = int(round(cov_p * len(y)))
    lo, hi = stats.binom.ppf([0.025, 0.975], len(y), 0.95)
    print(f"[1] GP（model1）の LOOCV：RMSE {res['gp']['rmse']:.2f}°")
    print(f"    95%予測区間（半幅の平均 {z * sp.mean():.1f}°）に入った数 {k}/{len(y)}"
          f"（ちょうど合っていれば {lo:.0f}〜{hi:.0f} 個に入るのが普通）")
    print(f"    95%信頼区間（平均の関数、半幅の平均 {z * sl.mean():.1f}°）に入った数 "
          f"{int(round(cov_l * len(y)))}/{len(y)} ← 1つの測定値を入れる区間ではない")

    # 2. 物理のベースライン：表面エネルギーから Neumann の式で平らな面の θ（学習なし）
    th = neumann_theta(s4["F_2"].to_numpy())
    res["neumann"] = metrics(y, th)
    print(f"[2] Young＋Neumann（F_2 だけ、学習なし）：RMSE {res['neumann']['rmse']:.2f}°・"
          f"MAE {res['neumann']['mae']:.2f}°・平均のずれ {np.mean(th - y):+.1f}°")
    # Wenzel：cosθ* = r cosθY から r を逆算し、粗さ Sa と関係があるかを見る（同じ側のときだけ）
    c_t, c_y = np.cos(np.radians(y)), np.cos(np.radians(th))
    ok = (np.sign(c_t) == np.sign(c_y)) & (np.abs(c_y) > 0.05)
    r_imp = c_t[ok] / c_y[ok]
    rho = stats.spearmanr(r_imp, s4["F_3"].to_numpy()[ok])
    res["wenzel_r"] = {"n": int(ok.sum()), "median": float(np.median(r_imp)),
                       "frac_below1": float(np.mean(r_imp < 1)), "spearman_sa": float(rho.statistic),
                       "p": float(rho.pvalue)}
    print(f"    Wenzel の r を逆算（{ok.sum()}種）：中央値 {np.median(r_imp):.2f}、"
          f"r が1未満（式の前提に反する）の割合 {np.mean(r_imp < 1):.2f}、"
          f"Sa との順位相関 {rho.statistic:+.2f}（p = {rho.pvalue:.2f}）")
    # 物理＋機械学習：Neumann の値からのずれだけを GP で学ぶ
    p_res, _ = loocv(lambda i: make_gp(), X1, y - th)
    res["neumann_plus_gp"] = metrics(y, th + p_res)
    print(f"    Neumann ＋ GP（ずれを学ぶ）の LOOCV：RMSE {res['neumann_plus_gp']['rmse']:.2f}°")

    # 3. 家で測る4種を学習から外して予測する（1つずつ外す LOOCV とは違い、4種まとめて外す）
    te = np.array([s4.index.get_loc(m) for m in HOME if m in s4.index])
    tr = np.setdiff1d(np.arange(len(y)), te)
    pred = {}
    for name, cols in (("model1", MODEL1), ("model2", MODEL2)):
        X = s4[cols].to_numpy()
        E = heldout_errors(X[tr], y[tr], SMALL)              # 設定は学習の22種の中だけで選ぶ
        best = SMALL[int(np.argmin((E ** 2).mean(0)))]
        m = MinMaxWrap(make_xgb(*best)).fit(X[tr], y[tr])
        pred[f"xgb_{name}"] = dict(zip(s4.index[te], m.predict(X[te]).round(2)))
        pred[f"xgb_{name}_setting"] = best
    gpm = make_gp().fit(X1[tr], y[tr])
    mu4, sd4 = gpm.predict(X1[te], return_std=True)
    pred["gp_model1"] = {mat: [float(mu4[j]), float(sd4[j])] for j, mat in enumerate(s4.index[te])}
    pred["neumann"] = dict(zip(s4.index[te], th[te].round(2)))
    pred["paper_label"] = dict(zip(s4.index[te], y[te]))
    res["leave_home_out"] = pred
    print("[3] 家の4種をまとめて外して学習（22種）→ 予測（表S4 の値）")
    for j, mat in enumerate(s4.index[te]):
        print(f"    {mat:5s} 表S4 {y[te][j]:5.1f}°  XGB-1 {pred['xgb_model1'][mat]:5.1f}°  "
              f"XGB-2 {pred['xgb_model2'][mat]:5.1f}°  GP {mu4[j]:5.1f}° ±{z * sd4[j]:4.1f}  Neumann {th[te][j]:5.1f}°")

    # 4. 家で測った値と比べる
    home_path = a.home
    if a.demo and home_path is None:
        home_path = str(Path("home") / "demo_angles_home.csv")
        truth = demo_home(home_path)
        print(f"[4] 練習用の「家の測定」を {home_path} に作りました（本物ではない。真の値 {truth}）")
    if home_path:
        h = read_home(home_path)
        res["home"] = h
        print("[4] 家の測定（片を単位にした平均と95%区間）とモデルの予測の差（家 − 予測）")
        for mat in HOME:
            if mat not in h or mat not in pred["gp_model1"]:
                continue
            g, s = pred["gp_model1"][mat]
            d = h[mat]["mean"] - g
            inside = abs(d) <= z * np.sqrt(s ** 2 + ((h[mat]["hi"] - h[mat]["lo"]) / 2 / z) ** 2)
            print(f"    {mat:5s} 家 {h[mat]['mean']:5.1f}° [{h[mat]['lo']:.1f}, {h[mat]['hi']:.1f}]"
                  f"（{h[mat]['pieces']}片・{h[mat]['drops']}滴）  GP との差 {d:+5.1f}°"
                  f"{'（区間の中）' if inside else '（区間の外）'}  表S4 との差 {h[mat]['mean'] - pred['paper_label'][mat]:+5.1f}°")

    save_json({"results": res, "gp_mu": mu, "gp_sd_pred": sp, "gp_sd_latent": sl, "y": y,
               "abbr": list(s4.index)}, OUT / "extend.json")
    plt = set_jp_font()
    fig, ax = plt.subplots(figsize=(9, 3.6))
    o = np.argsort(y)
    xx = np.arange(len(y))
    ax.errorbar(xx, mu[o], yerr=z * sp[o], fmt="o", color="C0", ms=4, capsize=2, label="GP の95%予測区間")
    ax.errorbar(xx + 0.25, mu[o], yerr=z * sl[o], fmt="none", color="C2", capsize=2, label="95%信頼区間")
    ax.scatter(xx, y[o], color="C3", marker="x", zorder=3, label="表S4 の値")
    ax.set_xticks(xx)
    ax.set_xticklabels(np.array(s4.index)[o], rotation=90, fontsize=7)
    ax.set_ylabel("WCA [°]")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "gp_intervals.png", dpi=150)
    if "home" in res:
        fig, ax = plt.subplots(figsize=(5, 3.6))
        mats = [m for m in HOME if m in res["home"]]
        xx = np.arange(len(mats))
        hm = np.array([res["home"][m]["mean"] for m in mats])
        herr = np.array([[hm[j] - res["home"][m]["lo"], res["home"][m]["hi"] - hm[j]] for j, m in enumerate(mats)]).T
        ax.errorbar(xx - 0.15, hm, yerr=herr, fmt="o", color="C3", capsize=3, label="家（片が単位の95%区間）")
        gm = np.array([pred["gp_model1"][m][0] for m in mats])
        gs = np.array([pred["gp_model1"][m][1] for m in mats])
        ax.errorbar(xx + 0.15, gm, yerr=z * gs, fmt="s", color="C0", capsize=3, label="GP（4種を外して学習）")
        ax.scatter(xx, [pred["paper_label"][m] for m in mats], marker="x", color="k", label="表S4 の値")
        ax.set_xticks(xx)
        ax.set_xticklabels(mats)
        ax.set_ylabel("WCA [°]")
        ax.legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(OUT / "home_vs_model.png", dpi=150)
    print(f"→ {OUT}/extend.json・gp_intervals.png（家の値があれば home_vs_model.png も）")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile synth_ymi.py
"""SI が手に入らないときの「代わりのデータ」（合成。答えが分かっている）を作る。

  uv run python synth_ymi.py                 # synth/sena_s4.csv・sena_s1.csv・truth.json
  uv run python synth_ymi.py --seed 3 --noise 5
列の形は本物の表S4 と同じ（F_1〜F_15・WCA）。値は本物ではない（高分子の名前を借りているだけ）。
真のしくみ：WCA = Neumann(F_2) + 5·(F_10 − 3) − (F_8 − 9) + 雑音（SD = --noise）。ほかの特徴は無関係。
truth.json には、同じしくみから作った新しい高分子 5000 個で測った「本当の汎化誤差」も入れる。
SI が手に入ったら、--data data（本物）に差し替えて同じスクリプトを動かす。
"""
import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd

from ymi_core import ABBR, MODEL1, MinMaxWrap, make_xgb, metrics, neumann_theta


def features(rng, n):
    F = {}
    F["F_2"] = np.clip(rng.normal(36, 7, n), 18, 48).round(0)
    F["F_3"] = np.exp(rng.normal(np.log(80), 1.0, n)).round(0)
    F["F_4"] = np.exp(rng.normal(np.log(140), 0.6, n)).round(0)
    heavy = rng.integers(6, 91, n)
    F["F_5"] = np.floor(heavy * rng.uniform(0, 0.6, n))
    F["F_6"] = heavy - F["F_5"]
    F["F_7"] = rng.uniform(0, 1, n).round(2)
    F["F_8"] = rng.integers(0, 19, n).astype(float)
    F["F_9"] = rng.integers(0, 7, n).astype(float)
    F["F_10"] = rng.normal(3.0, 1.5, n).round(2)
    F["F_11"] = heavy.astype(float)
    F["F_14"] = np.floor(heavy * rng.uniform(0.3, 1.2, n))
    F["F_13"] = (heavy * rng.uniform(11, 16, n)).round(0)
    F["F_12"] = F["F_13"] + F["F_14"]
    F["F_15"] = 4 * heavy + F["F_14"]
    return pd.DataFrame({c: F[c] for c in MODEL1})


def truth_fn(df):
    return neumann_theta(df["F_2"]) + 5.0 * (df["F_10"] - 3.0) - 1.0 * (df["F_8"] - 9.0)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--seed", type=int, default=1)
    ap.add_argument("--noise", type=float, default=5.0)
    ap.add_argument("--out", default="synth")
    a = ap.parse_args()
    rng = np.random.default_rng(a.seed)
    out = Path(a.out)
    out.mkdir(exist_ok=True)
    X = features(rng, 26)
    f = truth_fn(X)
    y = (f + rng.normal(0, a.noise, 26)).round(0)
    s4 = pd.concat([pd.Series(range(26), name="F_1"), X], axis=1)
    s4["WCA"] = y
    s4.to_csv(out / "sena_s4.csv", index=False)
    # 「自分たちで測った値」：19種だけ、真の値に別の雑音（SD 3°）
    idx = np.sort(rng.choice(26, 19, replace=False))
    s1 = pd.DataFrame({"abbr": [ABBR[i] for i in idx],
                       "wca_inhouse": (f[idx] + rng.normal(0, 3.0, 19)).round(0),
                       "wca_sd": 2.0, "sa_nm": X["F_3"].to_numpy()[idx]})
    s1.to_csv(out / "sena_s1.csv", index=False)
    # 本当の汎化誤差：同じしくみの新しい高分子 5000 個で測る
    Xn = features(rng, 5000)
    yn = truth_fn(Xn) + rng.normal(0, a.noise, 5000)
    true = {"noise_floor_rmse": a.noise}
    for s in [(50, 2, 0.1), (100, 6, 0.3), (20, 1, 0.7)]:
        m = MinMaxWrap(make_xgb(*s)).fit(X.to_numpy(), y)
        true[f"xgb_{s[0]}_{s[1]}_{s[2]}"] = metrics(yn, m.predict(Xn.to_numpy()))["rmse"]
    from reproduce_ymi import make_gp
    import warnings
    from sklearn.exceptions import ConvergenceWarning
    warnings.filterwarnings("ignore", category=ConvergenceWarning)
    gp = make_gp().fit(X.to_numpy(), y)
    mu, sd = gp.predict(Xn.to_numpy(), return_std=True)
    true["gp_model1"] = metrics(yn, mu)["rmse"]
    true["gp_cover95_fresh"] = float(np.mean(np.abs(yn - mu) <= 1.959964 * sd))
    truth = {"seed": a.seed, "noise_sd": a.noise,
             "formula": "Neumann(F_2) + 5*(F_10-3) - (F_8-9) + N(0, noise^2)",
             "relevant": ["F_2", "F_10", "F_8"], "true_generalization_rmse": true,
             "label_gap_expected_rmse": float(np.sqrt(a.noise ** 2 + 3.0 ** 2))}
    (out / "truth.json").write_text(json.dumps(truth, ensure_ascii=False, indent=1))
    print(f"{out}/sena_s4.csv（26行）・sena_s1.csv（19行）・truth.json を作りました")
    print(f"真の雑音 SD {a.noise}°。新しい5000個での本当の RMSE：" +
          "、".join(f"{k} {v:.2f}" for k, v in true.items()))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile polymers.csv
F_1,abbr,name_ja,repeat_smiles
0,PEEK,ポリエーテルエーテルケトン,*Oc1ccc(Oc2ccc(C(=O)c3ccc(*)cc3)cc2)cc1
1,PPS,ポリフェニレンスルフィド,*c1ccc(S*)cc1
2,PSU,ポリスルホン,*Oc1ccc(C(C)(C)c2ccc(Oc3ccc(S(=O)(=O)c4ccc(*)cc4)cc3)cc2)cc1
3,PC,ポリカーボネート,*Oc1ccc(C(C)(C)c2ccc(OC(*)=O)cc2)cc1
4,PE,ポリエチレン,*CC*
5,PP,ポリプロピレン,*CC(C)*
6,PET,ポリエチレンテレフタレート,*OCCOC(=O)c1ccc(C(*)=O)cc1
7,PMMA,ポリメタクリル酸メチル（アクリル）,*CC(C)(C(=O)OC)*
8,PEO,ポリエチレンオキシド,*CCO*
9,ABS,ABS樹脂（各1単位）,*CC(C#N)CC=CCCC(c1ccccc1)*
10,FEP,FEP（TFE＋HFP 各1単位）,*C(F)(F)C(F)(F)C(F)(F)C(F)(C(F)(F)F)*
11,NYLON6,ナイロン6,*NCCCCCC(*)=O
12,PBD,ポリブタジエン（1・4結合）,*CC=CC*
13,PCTFE,ポリクロロトリフルオロエチレン,*C(F)(F)C(F)(Cl)*
14,PIB,ポリイソブチレン,*CC(C)(C)*
15,PS,ポリスチレン,*CC(c1ccccc1)*
16,PTFE,ポリテトラフルオロエチレン,*C(F)(F)C(F)(F)*
17,PVA,ポリ酢酸ビニル,*CC(OC(C)=O)*
18,PVOH,ポリビニルアルコール,*CC(O)*
19,PVC,ポリ塩化ビニル,*CC(Cl)*
20,PVF,ポリフッ化ビニル,*CC(F)*
21,PVDF,ポリフッ化ビニリデン,*CC(F)(F)*
22,PES,ポリエーテルスルホン,*Oc1ccc(S(=O)(=O)c2ccc(*)cc2)cc1
23,PPO,ポリフェニレンオキシド,*Oc1c(C)cc(*)cc1C
24,SA,AS樹脂（各1単位）,*CC(c1ccccc1)CC(C#N)*
25,PBT,ポリブチレンテレフタレート,*OCCCCOC(=O)c1ccc(C(*)=O)cc1


## SI の PDF を用意する
自分のブラウザで落とした `jp4c06608_si_001.pdf` を選びます（YM-5 の手順6）。PDF がなければ、合成の代わりのデータで流れを練習します。Colab では、次のセルがアップロードの欄を出して待ちます（`USE_UPLOAD = False` にすると欄を出さずに進み、PDF がなければ合成データになります）。

In [ ]:
from pathlib import Path
Path("si").mkdir(exist_ok=True)
PDF = Path("si/jp4c06608_si_001.pdf")
USE_UPLOAD = True   # False にすると、アップロードの欄を出さずに進む（PDF がなければ合成データ。「すべてのセルを実行」で止まらない）
if IN_COLAB and USE_UPLOAD and not PDF.exists():
    from google.colab import files
    up = files.upload()
    for name, data in up.items():
        if name.endswith(".pdf"):
            PDF.write_bytes(data)
USE_SYNTH = not PDF.exists()
DATA, OUT = ("synth", "out_synth") if USE_SYNTH else ("data", "out")
print("合成データで練習します" if USE_SYNTH else "本物の表で再現します")

In [ ]:
import subprocess
from IPython.display import Image, display

def run(*args):
    r = subprocess.run([sys.executable, *args], capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr)
        raise RuntimeError(f"失敗：{args}")

if USE_SYNTH:
    run("synth_ymi.py")
else:
    run("si_to_csv.py", str(PDF))
    run("features_ymi.py", "--scan")
    run("features_ymi.py")

## 再現（表4・表S5・図3・重要度）

In [ ]:
run("reproduce_ymi.py", "--data", DATA, "--out", OUT)
display(Image(f"{OUT}/fig3_repro.png"))

## ずれの分析（入れ子の CV は最後の任意のセルで）

In [ ]:
run("deviation_ymi.py", "--data", DATA, "--out", OUT, "--skip-nested")
display(Image(f"{OUT}/cv_spread.png"))

## 拡張（ガウス過程の区間・物理のベースライン・家の4種）

In [ ]:
run("extend_ymi.py", "--data", DATA, "--out", OUT, "--demo")
display(Image(f"{OUT}/gp_intervals.png"))
display(Image(f"{OUT}/home_vs_model.png"))

## （任意）入れ子の交差検証
Mac の Apple シリコン で2〜6分ほど。Colab の CPU では、それより長くかかることがあります（要確認）。

In [ ]:
RUN_NESTED = False   # True にすると実行する
if RUN_NESTED:
    run("deviation_ymi.py", "--data", DATA, "--out", OUT)